# **Proyecto Etapa 3 — Aprendizaje Supervisado con PySpark**

### **Curso: TC5057 · Análisis de Grandes Volúmenes de Datos**
#### **Tecnológico de Monterrey**
##### **Profesor Titular: Dr. Iván Olmos Pineda**

---

**Actividad individual**

| Nombre | Matrícula |
|--------|-----------|
| Diego Falcón Costilla | A01139580 |

---
## 1. Introducción: Aprendizaje Supervisado

### 1.1 Concepto general

El **aprendizaje supervisado** es una rama del aprendizaje automático en la que un modelo es entrenado a partir de un conjunto de pares entrada–salida etiquetados $(x_i, y_i)$. El objetivo es aprender una función $f : X \rightarrow Y$ que generalice bien a datos no vistos, minimizando alguna función de pérdida sobre el conjunto de entrenamiento.

Formalmente, dado un conjunto de entrenamiento $\mathcal{D}_{\text{train}} = \{(x_1, y_1), \ldots, (x_n, y_n)\}$, el modelo aprende los parámetros $\theta^*$ tal que:

$$\theta^* = \arg\min_\theta \frac{1}{n} \sum_{i=1}^{n} \mathcal{L}(f_\theta(x_i),\, y_i)$$

donde $\mathcal{L}$ es la función de pérdida adecuada al problema (entropía cruzada para clasificación, error cuadrático para regresión).

Los dos grandes tipos de tareas supervisadas son:
- **Clasificación**: $Y$ es un conjunto discreto de clases (e.g., predecir el tipo de tejido).
- **Regresión**: $Y \subseteq \mathbb{R}$ (e.g., predecir un valor continuo de expresión génica).

---

### 1.2 Algoritmos representativos en la literatura

| Algoritmo | Tipo | Fortalezas | Limitaciones |
|-----------|------|------------|--------------|
| **Árbol de Decisión** (Decision Tree) | Clasificación / Regresión | Interpretable, no requiere normalización | Sobreajuste fácil; inestable ante pequeñas variaciones |
| **Random Forest** | Clasificación / Regresión | Robusto al sobreajuste; maneja alta dimensionalidad | Menos interpretable; costoso en memoria |
| **Gradient Boosted Trees (GBT)** | Clasificación / Regresión | Alta precisión; captura interacciones no lineales | Entrenamiento secuencial; más lento que RF |
| **Regresión Logística** | Clasificación | Simple, interpretable, probabilístico | Supone linealidad; sensible a multicolinealidad |
| **SVM (Support Vector Machine)** | Clasificación / Regresión | Efectivo en alta dimensión; margen máximo | Difícil de escalar a millones de instancias |
| **Perceptrón Multicapa (MLP)** | Clasificación / Regresión | Captura relaciones muy complejas | Requiere mucho dato y tunning cuidadoso |
| **Naive Bayes** | Clasificación | Muy rápido; bien calibrado con poco dato | Asume independencia entre features |

---

### 1.3 Algoritmos disponibles en PySpark MLlib

PySpark MLlib (`pyspark.ml.classification`) ofrece implementaciones distribuidas de los siguientes algoritmos de clasificación:

| Clase PySpark | Algoritmo |
|---------------|-----------|
| `DecisionTreeClassifier` | Árbol de decisión |
| `RandomForestClassifier` | Random Forest |
| `GBTClassifier` | Gradient Boosted Trees |
| `LogisticRegression` | Regresión logística (multinomial) |
| `MultilayerPerceptronClassifier` | Red neuronal MLP |
| `LinearSVC` | SVM lineal |
| `NaiveBayes` | Naive Bayes |
| `FMClassifier` | Factorization Machines |

El pipeline de MLlib sigue el patrón `Transformer → Estimator → Model`, compatible con `Pipeline` y `CrossValidator` para validación cruzada distribuida.

---

### 1.4 Algoritmo seleccionado: Random Forest

Se selecciona **Random Forest** por las siguientes razones aplicadas al contexto GTEx:

1. **Alta dimensionalidad**: cada muestra tiene miles de features (genes). RF selecciona aleatoriamente un subconjunto de features en cada split, lo que reduce la varianza sin requerir selección manual de genes.
2. **Robustez al sobreajuste**: el promedio de múltiples árboles decorrelacionados mitiga el sobreajuste que sufriría un árbol individual sobre datos de expresión génica ruidosos.
3. **Importancia de variables**: RF produce un ranking de importancia de genes (`featureImportances`), útil para interpretar qué genes distinguen los grupos de tejido.
4. **No requiere normalización estricta**: los valores TPM ya son comparables entre muestras, pero RF es insensible a escalas, a diferencia de MLP o SVM.

---
## 2. Selección de los datos

### 2.1 Estrategia

La muestra M del equipo (Etapa 2) abarca las 10 particiones P01–P10 construidas a partir del muestreo estratificado proporcional sobre el dataset GTEx V10. Para esta actividad individual se construye una sub-muestra **M'** con las siguientes restricciones:

- Se trabaja con las **particiones cardiovasculares y musculoesqueléticas** (P05, P06, P07, P08) para mantener el problema manejable.
- Se aplica un muestreo aleatorio adicional de **200 muestras por partición** (800 muestras totales), suficiente para entrenar un clasificador binario de sexo sin tiempos de procesamiento excesivos.
- La variable objetivo es **SEX_LABEL** (Masculino / Femenino), una tarea de clasificación binaria.
- Las features son los valores de expresión TPM de **500 genes** seleccionados aleatoriamente (columna por gen).

**Justificación de la variable objetivo:** predecir el sexo biológico a partir de la expresión génica es una tarea con respuesta conocida en la literatura — los genes del cromosoma Y (RPS4Y1, KDM5D, EIF1AY, etc.) son altamente expresados en tejidos masculinos y ausentes en femeninos. Esto permite verificar que el modelo aprende señal biológica real, no artefactos numéricos.

In [ ]:
import sys, os

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType

sys.path.insert(0, os.path.abspath('../src'))
from GlobalVariables import (
    FILE_PATH, SAMPLE_ATTRS_PATH, SUBJECT_PHENO_PATH,
    N_GENES, RANDOM_SEED
)

import random
import numpy as np
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sub-sample parameters
TARGET_TISSUE_GROUPS = ['Cardiovascular', 'Musculoesqueletico']
SAMPLES_PER_PARTITION = 200   # max samples taken from each of the 4 partitions
N_GENES_SUBSAMPLE     = 500   # number of randomly selected gene columns
GENE_STEP             = 100   # read every Nth column from the raw file first

print(f'Semilla aleatoria : {RANDOM_SEED}')
print(f'Grupos de tejido  : {TARGET_TISSUE_GROUPS}')
print(f'Muestras/partición: {SAMPLES_PER_PARTITION}')
print(f'Genes seleccionados: {N_GENES_SUBSAMPLE}')

In [ ]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('GTEx_SupervisedLearning_TC5057') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

spark.conf.set('spark.sql.repl.eagerEval.enabled', True)
spark

In [ ]:
# --- Load metadata ---
sa_df = spark.read.csv(SAMPLE_ATTRS_PATH, sep='\t', header=True) \
    .select('SAMPID', 'SMTS', 'SMTSD', 'SMAFRZE') \
    .filter(F.col('SMAFRZE') == 'RNASEQ') \
    .withColumn('SUBJID', F.regexp_extract(F.col('SAMPID'), r'^(GTEX-[^-]+)', 1))

sp_df = spark.read.csv(SUBJECT_PHENO_PATH, sep='\t', header=True) \
    .select('SUBJID', 'SEX')

meta_df = sa_df.join(sp_df, on='SUBJID', how='inner')

# Tissue group mapping (same as Etapa 2)
tissue_group_col = F.when(F.col('SMTS').isin('Brain', 'Nerve'), 'Nervioso') \
    .when(F.col('SMTS').isin('Blood', 'Bone Marrow', 'Spleen'), 'Hematopoyetico') \
    .when(F.col('SMTS').isin('Heart', 'Blood Vessel'), 'Cardiovascular') \
    .when(F.col('SMTS').isin('Muscle', 'Adipose Tissue', 'Skin'), 'Musculoesqueletico') \
    .otherwise('Visceral_Metabolico')

sex_label_col = F.when(F.col('SEX') == '1', 'Masculino').otherwise('Femenino')

meta_df = meta_df \
    .withColumn('TISSUE_GROUP', tissue_group_col) \
    .withColumn('SEX_LABEL', sex_label_col) \
    .withColumn('COL_NAME',
        F.regexp_replace(F.regexp_replace(F.col('SAMPID'), '-', '_'), '\\.', '_'))

# Filter to the two target tissue groups (P05, P06, P07, P08)
meta_target = meta_df.filter(F.col('TISSUE_GROUP').isin(TARGET_TISSUE_GROUPS))

print('Distribución por partición (target):')
meta_target.groupBy('TISSUE_GROUP', 'SEX_LABEL').count().orderBy('TISSUE_GROUP', 'SEX_LABEL').show()

In [ ]:
# --- Build M': stratified sub-sample, up to SAMPLES_PER_PARTITION per partition ---
# Collect sample IDs per (TISSUE_GROUP, SEX_LABEL) partition
from collections import defaultdict

meta_rows = meta_target.select('COL_NAME', 'TISSUE_GROUP', 'SEX_LABEL', 'SMTSD').collect()

# Group by partition
partition_samples = defaultdict(list)
for row in meta_rows:
    key = (row['TISSUE_GROUP'], row['SEX_LABEL'])
    partition_samples[key].append((row['COL_NAME'], row['SMTSD']))

# Stratified sampling within each partition: proportional to SMTSD subtypes
rng = random.Random(RANDOM_SEED)
selected_col_names = []
selected_meta = []   # [(col_name, tissue_group, sex_label)]

for (tg, sx), items in sorted(partition_samples.items()):
    n_partition = len(items)
    n_select = min(SAMPLES_PER_PARTITION, n_partition)

    # Group by SMTSD
    by_subtype = defaultdict(list)
    for col, smtsd in items:
        by_subtype[smtsd].append(col)

    # Proportional allocation
    sampled = []
    for smtsd, cols in by_subtype.items():
        n_strata = max(1, round(n_select * len(cols) / n_partition))
        n_strata = min(n_strata, len(cols))
        sampled.extend(rng.sample(cols, n_strata))

    # Trim to exact target if over-sampled due to rounding
    sampled = sampled[:n_select]

    for col in sampled:
        selected_col_names.append(col)
        selected_meta.append((col, tg, sx))

    print(f'  {tg:<22} + {sx:<12}: {len(sampled)} muestras seleccionadas de {n_partition}')

print(f'\nTotal M\' : {len(selected_col_names)} muestras')

In [ ]:
# --- Load TPM file columns for M' samples only ---
import pandas as pd
from pyspark.sql.functions import split as spark_split

# Read all column names
peek = pd.read_csv(FILE_PATH, sep='\t', skiprows=2, nrows=0)
all_col_names_raw = peek.columns.tolist()
all_col_names_clean = [c.replace('-', '_').replace('.', '_') for c in all_col_names_raw]

# Map clean name -> original index
name_to_idx = {clean: idx for idx, clean in enumerate(all_col_names_clean)}

# Find indices for Name, Description + selected samples
fixed_cols = ['Name', 'Description']
fixed_indices = [name_to_idx[c] for c in fixed_cols]

# Keep only selected sample columns that exist in the file
valid_sample_cols = [c for c in selected_col_names if c in name_to_idx]
sample_indices = [name_to_idx[c] for c in valid_sample_cols]

all_selected_indices = fixed_indices + sample_indices
all_selected_names   = fixed_cols + valid_sample_cols

print(f'Columnas de muestra válidas: {len(valid_sample_cols)} / {len(selected_col_names)}')

# Load only gene rows with selected columns via Spark
raw_df = spark.read.text(FILE_PATH)
gene_df = raw_df.filter(F.col('value').startswith('ENSG'))
split_col = spark_split(F.col('value'), '\t')

df_tpm = gene_df.select(
    *[split_col.getItem(i).alias(all_selected_names[idx])
      for idx, i in enumerate(all_selected_indices)]
)

for c in valid_sample_cols:
    df_tpm = df_tpm.withColumn(c, F.col(c).cast(FloatType()))

print(f'DataFrame TPM M\': {df_tpm.count():,} genes x {len(valid_sample_cols)} muestras')
df_tpm.select(all_selected_names[:5]).show(3)

In [ ]:
# --- Select N_GENES_SUBSAMPLE genes randomly to keep feature matrix manageable ---
gene_ids = [row['Name'] for row in df_tpm.select('Name').collect()]
selected_gene_ids = rng.sample(gene_ids, min(N_GENES_SUBSAMPLE, len(gene_ids)))
selected_gene_ids_set = set(selected_gene_ids)

df_tpm_sub = df_tpm.filter(F.col('Name').isin(selected_gene_ids_set))
print(f'Genes seleccionados: {df_tpm_sub.count()} de {len(gene_ids):,}')

In [ ]:
# --- Pivot: rows=samples, columns=genes (transpose the TPM matrix) ---
# Collect the gene x sample matrix to pandas for pivoting (small enough: 500 genes x ~800 samples)
tpm_pd = df_tpm_sub.select(['Name'] + valid_sample_cols).toPandas()
tpm_pd = tpm_pd.set_index('Name')

# Transpose: rows=samples, columns=genes
tpm_T = tpm_pd.T.reset_index()
tpm_T = tpm_T.rename(columns={'index': 'COL_NAME'})

# Attach labels from selected_meta
meta_dict = {col: (tg, sx) for col, tg, sx in selected_meta}
tpm_T['TISSUE_GROUP'] = tpm_T['COL_NAME'].map(lambda c: meta_dict.get(c, (None, None))[0])
tpm_T['SEX_LABEL']    = tpm_T['COL_NAME'].map(lambda c: meta_dict.get(c, (None, None))[1])
tpm_T = tpm_T.dropna(subset=['TISSUE_GROUP', 'SEX_LABEL'])

# Replace NaN TPM values with 0 (unexpressed genes)
gene_cols = selected_gene_ids
tpm_T[gene_cols] = tpm_T[gene_cols].fillna(0.0)

print(f'Matriz M\' final: {tpm_T.shape[0]} muestras x {len(gene_cols)} features')
print('Distribución de clases:')
print(tpm_T['SEX_LABEL'].value_counts())

---
## 3. Preparación del conjunto de entrenamiento y prueba

### 3.1 Técnica de división

Se aplica una **división estratificada 80 / 20** (train / test):

- **80% entrenamiento**: proporciona suficiente dato para que Random Forest construya árboles robustos con alta dimensionalidad (500 features).
- **20% prueba**: conjunto independiente para estimar el error de generalización. Con ~800 muestras, el 20% equivale a ~160 instancias — suficiente para obtener métricas estadísticamente estables en clasificación binaria.

**Por qué estratificada:** se mantiene la misma proporción de clases (Masculino / Femenino) en entrenamiento y prueba. Sin estratificación, una división aleatoria simple podría concentrar más muestras de un sexo en prueba, sesgando las métricas de evaluación. La estratificación también es consistente con la técnica usada en Etapa 2 para construir M.

**Semilla fija (`RANDOM_SEED = 42`):** garantiza que la división sea reproducible — cualquier ejecución del notebook genera el mismo train/test.

In [ ]:
# --- Encode label: Masculino=1, Femenino=0 ---
tpm_T['label'] = tpm_T['SEX_LABEL'].map({'Masculino': 1, 'Femenino': 0}).astype(int)

print('Codificación de la variable objetivo:')
print(tpm_T[['SEX_LABEL', 'label']].value_counts())

In [ ]:
# --- Stratified 80/20 split ---
from sklearn.model_selection import train_test_split

feature_cols = gene_cols
X = tpm_T[feature_cols].values
y = tpm_T['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y
)

print(f'Entrenamiento : {X_train.shape[0]} muestras  ({X_train.shape[0]/len(y)*100:.1f}%)')
print(f'Prueba        : {X_test.shape[0]} muestras  ({X_test.shape[0]/len(y)*100:.1f}%)')
print(f'\nClases en entrenamiento : Masculino={y_train.sum()}, Femenino={len(y_train)-y_train.sum()}')
print(f'Clases en prueba        : Masculino={y_test.sum()}, Femenino={len(y_test)-y_test.sum()}')

In [ ]:
# --- Convert to Spark DataFrames for MLlib ---
import pandas as pd

train_pd = pd.DataFrame(X_train, columns=feature_cols)
train_pd['label'] = y_train.tolist()

test_pd = pd.DataFrame(X_test, columns=feature_cols)
test_pd['label'] = y_test.tolist()

train_spark = spark.createDataFrame(train_pd)
test_spark  = spark.createDataFrame(test_pd)

print(f'Spark train: {train_spark.count()} filas x {len(feature_cols)+1} columnas')
print(f'Spark test : {test_spark.count()} filas x {len(feature_cols)+1} columnas')

---
## 4. Construcción de modelos de aprendizaje supervisado

### 4.1 Variable objetivo y tarea

**Variable objetivo:** `SEX_LABEL` (Masculino / Femenino) — clasificación binaria.

**Justificación:** La expresión génica del cromosoma Y es una señal biológica altamente discriminativa entre sexos (genes como *RPS4Y1*, *KDM5D*, *DDX3Y* tienen TPM ≈ 0 en mujeres y valores altos en hombres). Esto permite evaluar si el modelo captura señal real. Además, el desbalance de clases en GTEx (≈67% masculino, ≈33% femenino) es moderado y manejable sin técnicas de re-muestreo adicionales.

### 4.2 Pipeline MLlib

El pipeline sigue tres pasos:
1. **`VectorAssembler`**: convierte las columnas de genes en un único vector de features denso.
2. **`RandomForestClassifier`**: entrena el modelo con los hiperparámetros seleccionados.
3. **Evaluación**: se mide con **AUC-ROC** (Area Under the ROC Curve) como métrica principal, complementada con la matriz de confusión y el `accuracy`.

**Justificación de AUC-ROC como métrica:** Es invariante al umbral de decisión y robusta ante clases desbalanceadas. Un valor de 1.0 indica clasificación perfecta; 0.5 equivale a clasificación aleatoria. Para datos biomédicos es el estándar recomendado cuando existe desbalance de clases.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# Step 1: assemble gene expression columns into a single feature vector
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features'
)

# Step 2: Random Forest
# - numTrees=100: balance between accuracy and compute time
# - maxDepth=10: prevents overfitting on the 500-gene feature space
# - featureSubsetStrategy='sqrt': standard for classification (sqrt of 500 ≈ 22 genes per split)
# - seed=RANDOM_SEED: reproducibility
rf = RandomForestClassifier(
    labelCol='label',
    featuresCol='features',
    numTrees=100,
    maxDepth=10,
    featureSubsetStrategy='sqrt',
    seed=RANDOM_SEED
)

pipeline = Pipeline(stages=[assembler, rf])
print('Pipeline configurado:', [s.__class__.__name__ for s in pipeline.getStages()])

In [ ]:
# --- Train ---
print('Entrenando Random Forest...')
model = pipeline.fit(train_spark)
print('Entrenamiento completado.')

In [ ]:
# --- Predict on test set ---
predictions = model.transform(test_spark)
predictions.select('label', 'prediction', 'probability').show(10)

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# AUC-ROC (primary metric)
auc_evaluator = BinaryClassificationEvaluator(
    labelCol='label',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)
auc = auc_evaluator.evaluate(predictions)

# Accuracy (secondary metric)
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='accuracy'
)
accuracy = acc_evaluator.evaluate(predictions)

# F1-score
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='f1'
)
f1 = f1_evaluator.evaluate(predictions)

print('=' * 40)
print(f'  AUC-ROC  : {auc:.4f}')
print(f'  Accuracy : {accuracy:.4f}')
print(f'  F1-Score : {f1:.4f}')
print('=' * 40)

In [ ]:
# --- Confusion matrix ---
cm_pd = predictions.select('label', 'prediction').toPandas()
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(cm_pd['label'], cm_pd['prediction'])
print('Matriz de confusión:')
print('                Pred Femenino  Pred Masculino')
print(f'Real Femenino   {cm[0,0]:>12}  {cm[0,1]:>13}')
print(f'Real Masculino  {cm[1,0]:>12}  {cm[1,1]:>13}')
print()
print(classification_report(cm_pd['label'], cm_pd['prediction'],
                             target_names=['Femenino', 'Masculino']))

In [ ]:
# --- Top 20 most important genes ---
rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray()

feat_imp = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)

print('Top 20 genes más importantes para clasificar el sexo:')
print(f'{"Rank":<6} {"Gene ID":<20} {"Importancia"}')
print('-' * 40)
for rank, (gene, imp) in enumerate(feat_imp[:20], 1):
    print(f'{rank:<6} {gene:<20} {imp:.5f}')

---
### 4.3 Interpretación de resultados

#### Métricas de evaluación

| Métrica | Valor esperado | Interpretación |
|---------|----------------|----------------|
| **AUC-ROC** | > 0.95 | Clasificación binaria de sexo a partir de expresión génica es una tarea bien separable biológicamente. Valores > 0.95 son esperables dado que genes del cromosoma Y son altamente discriminativos. |
| **Accuracy** | > 0.90 | Con 500 genes aleatorios es probable que algunos genes sexo-específicos sean incluidos, garantizando alta precisión. |
| **F1-Score** | > 0.90 | Confirma que el modelo no sacrifica recall por precision ni viceversa, especialmente relevante con el leve desbalance de clases (≈67/33). |

#### Genes más importantes

Se espera que los genes de mayor importancia incluyan genes del cromosoma Y como **RPS4Y1**, **KDM5D**, **DDX3Y**, **EIF1AY** y **USP9Y**. Estos genes codifican proteínas esenciales pero tienen copias únicas en el cromosoma Y — ausentes por definición en el genoma femenino XX, lo que los convierte en marcadores perfectos del sexo biológico en datos de expresión.

La presencia de estos genes en el top-20 valida que el modelo aprende **señal biológica real** y no sobreajusta a artefactos del muestreo.

#### Supuestos del modelo

1. **Independencia entre instancias**: se asume que cada muestra de tejido es independiente. En GTEx, un donante puede contribuir múltiples tejidos — si dos tejidos del mismo donante caen en el mismo subconjunto, se introduce una correlación leve. Este efecto es pequeño con la división estratificada aplicada.
2. **Valores TPM como proxies de expresión**: se usan valores TPM sin transformación logarítmica. RF es invariante a escala, pero la distribución fuertemente sesgada (muchos ceros, pocos genes con TPM alto) podría favorecer genes constitutivamente activos. Para análisis más fino se recomendaría `log1p(TPM)`.
3. **Genes seleccionados aleatoriamente**: la selección aleatoria de 500 genes garantiza que no se introduce sesgo de conocimiento previo, pero incluye genes informativos y no informativos. Un segundo experimento con selección basada en varianza o información mutua podría mejorar la eficiencia del modelo.
4. **Particiones Cardiovascular + Musculoesquelético**: los resultados son representativos de estos tejidos pero no necesariamente generalizables a tejidos nerviosos (donde la expresión diferencial de sexo puede ser menos pronunciada).

---
## Referencias

1. Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324
2. GTEx Consortium. (2020). The GTEx Consortium atlas of genetic regulatory effects across human tissues. *Science*. https://doi.org/10.1126/science.aaz1776
3. Mank, J. E. (2017). Sex chromosomes and the evolution of sexual dimorphism. *Evolution*, 71(1), 162–172.
4. Apache Spark MLlib. (2024). Classification and regression. https://spark.apache.org/docs/latest/ml-classification-regression.html
5. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825–2830.

---

## Declaración de uso de Inteligencia Artificial

Anthropic. (2026). *Claude Sonnet 4.6* [Modelo de lenguaje grande], utilizado para soporte en estructura del notebook, documentación de celdas markdown y revisión del código PySpark. https://claude.ai

*La responsabilidad final sobre el contenido entregado recae en el autor. Las decisiones de diseño del experimento, selección de algoritmo, variable objetivo y la interpretación de resultados son del autor.*